# TE01 - OpenCV Basics, Python Practice, and 2D Discrete Fourier Transform

## Course Intent and Scope

TE01 has three major goals:
1. Get comfortable with OpenCV + NumPy + Matplotlib for basic image I/O and display.
2. Compute, visualize, and interpret 2D DFT amplitude spectra.
3. Build a first webcam capture + display + save workflow.

## Recommended Documentation Links

- OpenCV docs: http://docs.opencv.org/3.2.0/
- OpenCV-Python tutorial root: http://docs.opencv.org/3.2.0/d6/d00/tutorial_py_root.html
- OpenCV image display tutorial: https://docs.opencv.org/3.2.0/dc/d2e/tutorial_py_image_display.html
- NumPy docs: http://www.numpy.org/
- SciPy docs: https://docs.scipy.org/doc/
- Matplotlib docs: http://matplotlib.org/


## Environment Setup

Expected dataset files:
- `../data/te01/voilier_oies_blanches.jpg`
- `../data/te01/bibliotheque.jpg`
- `../data/te01/crayons.jpg`
- `../data/te01/D1.tif`
- `../data/te01/D4.tif`


In [ ]:
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path('../data/te01')
OUT_DIR = Path('../outputs/te01')
OUT_DIR.mkdir(parents=True, exist_ok=True)

required_files = [
    'voilier_oies_blanches.jpg',
    'bibliotheque.jpg',
    'crayons.jpg',
    'D1.tif',
    'D4.tif',
]

missing = [f for f in required_files if not (DATA_DIR / f).exists()]
if missing:
    raise FileNotFoundError(f'Missing files: {missing}')


def show_gray(image, title, *, xlabel='x (pixels)', ylabel='y (pixels)'):
    plt.figure(figsize=(6, 4))
    plt.imshow(image, cmap='gray')
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()


def compute_spectrum(image, shape=(), shift=False):
    spectrum_shape = shape if shape else None
    spectrum = np.fft.fft2(image, s=spectrum_shape)
    if shift:
        spectrum = np.fft.fftshift(spectrum)
    amplitude = np.abs(spectrum)
    log_amplitude = np.log2(1.0 + amplitude)
    return spectrum, amplitude, log_amplitude


def show_spectrum(log_amplitude, title, *, shifted=False):
    extent = [-0.5, 0.5, 0.5, -0.5] if shifted else [0.0, 1.0, 1.0, 0.0]
    plt.figure(figsize=(6, 4))
    plt.imshow(log_amplitude, cmap='magma', extent=extent, aspect='auto')
    plt.title(title)
    plt.xlabel('u')
    plt.ylabel('v')
    plt.tight_layout()
    plt.show()


def run_camera_session(out_dir, window_name, *, allow_snapshot=False, record_path='', transform=lambda frame: frame):
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print('Webcam unavailable on index 0.')
        return

    snapshot_idx = 1
    writer = cv2.VideoWriter()

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                print('Camera stream ended.')
                break

            display_frame = transform(frame)
            if record_path and not writer.isOpened():
                height, width = display_frame.shape[:2]
                is_color = display_frame.ndim == 3
                fourcc = cv2.VideoWriter_fourcc(*'XVID')
                writer.open(str(record_path), fourcc, 20.0, (width, height), isColor=is_color)

            cv2.imshow(window_name, display_frame)
            if writer.isOpened():
                writer.write(display_frame)

            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                break
            if allow_snapshot and key == ord('s'):
                snapshot_path = out_dir / f'fra{snapshot_idx}.png'
                cv2.imwrite(str(snapshot_path), display_frame)
                print(f'Saved {snapshot_path.name}')
                snapshot_idx += 1
    finally:
        cap.release()
        if writer.isOpened():
            writer.release()
        cv2.destroyAllWindows()


print('Setup OK')


## 1. Familiarization with Core Tools

### Task 1.1 - Read a grayscale image

Read `voilier_oies_blanches.jpg` in grayscale and call it `IMG`.

In [ ]:
fname = "voilier_oies_blanches.jpg"
image_path = DATA_DIR / fname

img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)

plt.imshow(img, cmap="grey")
plt.axis("off")
plt.show()

### Task 1.2 - Print image metadata in console

Print:
- image width and height (in pixels)
- quantization info (bytes per pixel value)
- pixel dtype
- dimensionality

In [ ]:
H, W = img.shape
print(f'Width: {W} px')
print(f'Height: {H} px')
print(f'Quantization: {img.itemsize} byte(s) per pixel')
print(f'dtype: {img.dtype}')
print(f'Dimensions: {img.ndim}')


### Task 1.3 - Display IMG with two toolchains

Display `IMG`:
1. with OpenCV window functions
2. with Matplotlib

Useful OpenCV functions from statement:
- `namedWindow`, `imshow`, `waitKey(0)`, `destroyWindow`, `destroyAllWindows`

`waitKey(delay)` note:
- `delay=0` waits indefinitely for key press and returns key code.


In [ ]:
try:
    cv2.namedWindow('TE01 - IMG', cv2.WINDOW_NORMAL)
    cv2.imshow('TE01 - IMG', img)
    cv2.waitKey(250)
    cv2.destroyWindow('TE01 - IMG')
    print('OpenCV display completed.')
except cv2.error as exc:
    print(f'OpenCV window skipped: {exc}')

plt.figure(figsize=(6, 4))
plt.imshow(img, cmap='gray')
plt.title('IMG displayed with Matplotlib')
plt.xlabel('x (pixels)')
plt.ylabel('y (pixels)')
plt.tight_layout()
plt.show()


## 2. 2D Discrete Fourier Transform

You will analyze amplitude spectra for different spatial patterns.

Main images for interpretation:
- `D1.tif`
- `D4.tif`
- `crayons.jpg`
- `bibliotheque.jpg`


### 2.1 DFT Computation and Visualization

Tasks to complete (from statement):
1. Read `bibliotheque.jpg` and convert to grayscale.
2. Visualize it with title and axis labels.
3. Compute 2D DFT using `numpy.fft.fft2`.
4. Compute amplitude spectrum `|F(u,v)|` for spatial frequencies in `[0, 1]`.
5. Visualize preferably `log2(1 + |F(u,v)|)` with referenced axes.
6. Use `fftshift` to represent frequencies in `[-0.5, 0.5]`.
7. Visualize shifted spectrum with referenced axes.
8. Explain why shifted display improves interpretation.
9. Compute a higher-resolution spectrum (e.g., 1024x1024) and comment.
   Also answer: what is this operation called?


In [ ]:
img_bib_path = DATA_DIR / 'bibliotheque.jpg'
img_bib = cv2.imread(str(img_bib_path), cv2.IMREAD_GRAYSCALE)
if img_bib is None:
    raise FileNotFoundError(f'Unable to read {img_bib_path}')

show_gray(img_bib, 'bibliotheque.jpg in grayscale')


In [ ]:
F_bib, amplitude_bib, log_amplitude_bib = compute_spectrum(img_bib)
show_spectrum(log_amplitude_bib, 'Log amplitude spectrum of bibliotheque.jpg', shifted=False)

print(f'Spectrum shape: {F_bib.shape}')
print(f'Amplitude range: {amplitude_bib.min():.2f} to {amplitude_bib.max():.2f}')


In [ ]:
F_bib_shifted = np.fft.fftshift(F_bib)
log_amplitude_bib_shifted = np.log2(1.0 + np.abs(F_bib_shifted))

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.imshow(log_amplitude_bib, cmap='magma', extent=[0.0, 1.0, 1.0, 0.0], aspect='auto')
plt.title('Non-shifted spectrum')
plt.xlabel('u')
plt.ylabel('v')

plt.subplot(1, 2, 2)
plt.imshow(log_amplitude_bib_shifted, cmap='magma', extent=[-0.5, 0.5, 0.5, -0.5], aspect='auto')
plt.title('Shifted spectrum')
plt.xlabel('u')
plt.ylabel('v')
plt.tight_layout()
plt.show()


### Interpretation Notes

- `u` and `v` are the horizontal and vertical spatial frequencies of the image.
- Normalized frequencies in `[0, 1]` make spectra comparable even when image sizes differ.
- `fftshift` moves the low-frequency energy to the center, which makes symmetry and dominant directions easier to read.
- Sampling the spectrum on a larger grid is zero-padding: it densifies the sampled spectrum without creating new frequencies.


In [ ]:
_, _, log_amplitude_bib_1024 = compute_spectrum(img_bib, shape=(1024, 1024), shift=True)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.imshow(log_amplitude_bib_shifted, cmap='magma', extent=[-0.5, 0.5, 0.5, -0.5], aspect='auto')
plt.title('Shifted spectrum on native grid')
plt.xlabel('u')
plt.ylabel('v')

plt.subplot(1, 2, 2)
plt.imshow(log_amplitude_bib_1024, cmap='magma', extent=[-0.5, 0.5, 0.5, -0.5], aspect='auto')
plt.title('Shifted spectrum after zero-padding to 1024x1024')
plt.xlabel('u')
plt.ylabel('v')
plt.tight_layout()
plt.show()

print('Zero-padding gives a smoother spectral display because it samples the same spectrum more densely.')


### 2.2 Interpretation Tasks on Multiple Images

Tasks from statement:
1. Display `D1.tif` and `D4.tif`.
2. Compute DFT + amplitude spectra for both.
3. Explain observed spectra.
4. Discuss whether D1 and D4 can be distinguished using Fourier transform.
5. Display grayscale `crayons.jpg` and `bibliotheque.jpg`.
6. Compute DFT + amplitude spectra for both.
7. Interpret spectra according to spatial content and theory.


In [ ]:
img_d1 = cv2.imread(str(DATA_DIR / 'D1.tif'), cv2.IMREAD_GRAYSCALE)
img_d4 = cv2.imread(str(DATA_DIR / 'D4.tif'), cv2.IMREAD_GRAYSCALE)

if img_d1 is None or img_d4 is None:
    raise FileNotFoundError('Unable to read D1.tif or D4.tif')

_, _, log_d1 = compute_spectrum(img_d1, shift=True)
_, _, log_d4 = compute_spectrum(img_d4, shift=True)

plt.figure(figsize=(12, 8))
for idx, (image, spectrum, title) in enumerate([
    (img_d1, log_d1, 'D1'),
    (img_d4, log_d4, 'D4'),
]):
    plt.subplot(2, 2, 2 * idx + 1)
    plt.imshow(image, cmap='gray')
    plt.title(f'{title} image')
    plt.axis('off')

    plt.subplot(2, 2, 2 * idx + 2)
    plt.imshow(spectrum, cmap='magma', extent=[-0.5, 0.5, 0.5, -0.5], aspect='auto')
    plt.title(f'{title} shifted spectrum')
    plt.xlabel('u')
    plt.ylabel('v')

plt.tight_layout()
plt.show()


In [ ]:
img_crayons = cv2.imread(str(DATA_DIR / 'crayons.jpg'), cv2.IMREAD_GRAYSCALE)
if img_crayons is None:
    raise FileNotFoundError('Unable to read crayons.jpg')

_, _, log_crayons = compute_spectrum(img_crayons, shift=True)
_, _, log_bib_shifted = compute_spectrum(img_bib, shift=True)

plt.figure(figsize=(12, 8))
for idx, (image, spectrum, title) in enumerate([
    (img_crayons, log_crayons, 'crayons'),
    (img_bib, log_bib_shifted, 'bibliotheque'),
]):
    plt.subplot(2, 2, 2 * idx + 1)
    plt.imshow(image, cmap='gray')
    plt.title(f'{title} image')
    plt.axis('off')

    plt.subplot(2, 2, 2 * idx + 2)
    plt.imshow(spectrum, cmap='magma', extent=[-0.5, 0.5, 0.5, -0.5], aspect='auto')
    plt.title(f'{title} shifted spectrum')
    plt.xlabel('u')
    plt.ylabel('v')

plt.tight_layout()
plt.show()


### Interpretation Notes

- `D1` and `D4` both generate sparse spectral peaks, which confirms the presence of repeated geometric patterns.
- They can still be distinguished in the Fourier domain because the dominant peak orientations and spacings differ.
- `crayons.jpg` concentrates energy along a few strong directions linked to aligned edges, while `bibliotheque.jpg` spreads energy over more orientations and scales.
- Regular spatial structures create concentrated peaks, whereas richer textures produce a broader and denser spectrum.


## 3. Webcam Acquisition, Visualization, and Recording

From statement, this part should be done at the end.

Tasks:
1. Acquire and visualize webcam stream in a window.
2. Choose a key to stop execution.
3. Add key-driven snapshot save (`fra1`, `fra2`, `fra3`, ...).
4. Acquire and save full video stream using `VideoWriter`.

Useful functions:
- `cv2.VideoCapture`, `read`, `release`, `cv2.waitKey`
- `cv2.VideoWriter_fourcc`, `cv2.VideoWriter`, `write`


In [ ]:
RUN_VIDEO_DEMOS = False

if RUN_VIDEO_DEMOS:
    run_camera_session(OUT_DIR, 'TE01 Preview')
else:
    print("Set RUN_VIDEO_DEMOS = True to run the webcam preview. Press 'q' to quit.")


In [ ]:
RUN_VIDEO_DEMOS = False

if RUN_VIDEO_DEMOS:
    run_camera_session(OUT_DIR, 'TE01 Preview + Snapshots', allow_snapshot=True)
else:
    print("Set RUN_VIDEO_DEMOS = True to enable snapshots. Press 's' to save fra1.png, fra2.png, ...")


In [ ]:
RUN_VIDEO_DEMOS = False
record_path = OUT_DIR / 'te01_webcam.avi'

if RUN_VIDEO_DEMOS:
    run_camera_session(OUT_DIR, 'TE01 Recording', record_path=record_path)
else:
    print(f"Set RUN_VIDEO_DEMOS = True to record the webcam stream to {record_path.name}.")
